[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/validation/labse-multilingual-smoketest/experiments/labse_english_hebrew_only.ipynb)


**1. Imports.** Install the public model package if needed and import all libraries.


In [ ]:
# 1. All imports and the single optional dependency installation.
import hashlib
import importlib.util
import io
import re
import subprocess
import sys
from urllib.request import urlopen

if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "sentence-transformers==5.1.2",
    ])

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer


**2. Model.** Load the pretrained LaBSE encoder on CPU.


In [ ]:
# 2. Download/cache LaBSE once. No API token, SSH, or paid GPU.
model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
model.eval()
print("LaBSE loaded.")


**3. Data.** Load and verify the full survey and metadata; display all 200 records.


In [ ]:
# 3. Download the complete survey and tagged metadata from a fixed GitHub commit.
# This is the HTTPS + SHA-256 method used in your older notebook; no SSH key is needed.
SOURCE_ROOT = (
    "https://raw.githubusercontent.com/shulruferor-sudo/"
    "Advance-Data-Science/81ae343dd70fb4165a64fd498f5ea0b2c6c86a73/"
    "Datasets/cbs_occupation_survey_sample/"
)

def load_csv(filename, sha256):
    with urlopen(SOURCE_ROOT + filename, timeout=60) as response:
        content = response.read()
    assert hashlib.sha256(content).hexdigest() == sha256, filename
    return pd.read_csv(io.BytesIO(content), dtype="string", keep_default_na=False)

survey_df = load_csv(
    "survey_200_records_current.csv",
    "c4928eedc97552f080968f816e1166e4ee4af1cda1aef42aab041d606a5b8625",
)
metadata_df = load_csv(
    "survey_metadata_he_en_names.csv",
    "bf2d5034b777007bcdad5c545f70e90720dc220647fc5d0f1487740733d78c3e",
)

# Select columns by their metadata label N, not by a hardcoded list.
name_columns = metadata_df.loc[
    metadata_df["HE_EN_NAMES"].eq("N"), "משתנה"
].tolist()

assert survey_df.shape == (200, 18)
assert survey_df["ID"].is_unique
assert len(name_columns) == 7
assert set(name_columns).issubset(survey_df.columns)

print("Survey:", survey_df.shape, "| English/Hebrew fields:", name_columns)
with pd.option_context("display.max_rows", 220, "display.max_columns", 18):
    display(survey_df)


**4. Inference.** In columns marked `N`, propose only the highest-scoring existing Hebrew name when cosine > α; keep the original data unchanged.


In [ ]:
# 4. Inference only: the best existing Hebrew label per English expression.
ALPHA = 0.50

# Detect English words, acronyms, names, and phrases inside any cell.
LATIN = re.compile(
    r"[A-Za-z]+(?:[&./'-][A-Za-z]+)*(?:\s+[A-Za-z]+(?:[&./'-][A-Za-z]+)*)*"
)
HEBREW = re.compile(r"[\u0590-\u05FF]")

# Preserve the survey. Only the copy receives proposed substitutions.
proposed_df = survey_df.copy()
scores = []
changed_cells = []

for column in name_columns:
    values = survey_df[column].fillna("").astype(str)
    english = sorted({m.group() for value in values for m in LATIN.finditer(value)})
    hebrew = sorted({
        value.strip() for value in values
        if HEBREW.search(value) and not LATIN.search(value) and value.strip()
    })
    if not english or not hebrew:
        continue

    # Compare English against Hebrew ONLY within the same field.
    vectors = model.encode(
        english + hebrew, batch_size=8, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    )
    similarities = vectors[:len(english)] @ vectors[len(english):].T
    selected = {}

    for i, term in enumerate(english):
        top = int(np.argmax(similarities[i]))
        score = float(similarities[i, top])
        candidate = hebrew[top]
        if score > ALPHA:
            selected[term] = candidate

        scores.append({
            "Column": column, "English": term, "Best_Hebrew": candidate,
            "Cosine": round(score, 4), "Above_alpha": score > ALPHA,
        })

    # Inspect every populated cell, including mixed Hebrew/English text.
    for index, original in values.items():
        if not LATIN.search(original):
            continue
        proposed = LATIN.sub(
            lambda match: selected.get(match.group(), match.group()), original,
        )
        proposed_df.at[index, column] = proposed
        changed_cells.append({
            "ID": survey_df.at[index, "ID"], "Column": column,
            "Original": original, "Proposed": proposed,
            "Changed": proposed != original,
        })

# Review the highest candidate for each English expression and every affected cell.
scores_df = pd.DataFrame(scores).sort_values(
    ["Column", "Cosine"], ascending=[True, False],
)
changes_df = pd.DataFrame(changed_cells)
print("Highest Hebrew candidate per English expression (α =", ALPHA, ")")
display(scores_df)
print("All cells containing English; changes are proposals only:")
display(changes_df)
print("Proposed DataFrame:")
with pd.option_context("display.max_rows", 220, "display.max_columns", 18):
    display(proposed_df)
